# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [2]:
# imports
import os
from openai import OpenAI
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv

In [ ]:
# constants
load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY")

if api_key is not None:
    print("API key is set up correctly!")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

MODEL_GPT = 'gemini-2.5-flash'
MODEL_LLAMA = 'llama3.2'


openai = OpenAI(api_key=api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

API key is set up correctly!


In [4]:
# set up environment
system_prompt = "You are an assistant to answer the questions. \
    Answer the question in an proper format with relevant information.\
        Answer should be proper formatted as markdown."

In [5]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [9]:
# Get gemini-2.5-flash to answer, with streaming
def answer():
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

print(answer())

This line of code is a powerful and concise way to extract unique, non-empty author names from a collection of book dictionaries, specifically designed to be used within a Python [generator function](https://docs.python.org/3/howto/generators.html).

Let's break it down:

## What this code does:

1.  **`for book in books`**: It iterates over an assumed iterable called `books`. We can infer `books` is a collection (like a list) of dictionaries, where each dictionary represents a book.

2.  **`book.get("author")`**: For each `book` dictionary, it attempts to retrieve the value associated with the key `"author"`.
    *   Using `.get()` is safer than `book["author"]` because if the `"author"` key is missing from a `book` dictionary, `.get()` will return `None` instead of raising a `KeyError`.

3.  **`if book.get("author")`**: This is a conditional filter. It checks if the value returned by `book.get("author")` is "truthy".
    *   In Python, `None` and empty strings (`""`) are considered "falsy". Any non-empty string (like an author's name) is "truthy".
    *   Therefore, this condition ensures that only books with an actual, non-empty author name are considered. Books with a missing author key (resulting in `None`) or an empty string as an author will be skipped.

4.  **`{ ... for ... if ... }`**: This entire construct is a [set comprehension](https://docs.python.org/3/tutorial/datastructures.html#sets).
    *   It builds a new `set` where each element is the `book.get("author")` value that passed the `if` condition.
    *   **Key Feature of Sets:** Sets inherently store only unique elements. If multiple books share the same author, that author's name will only appear once in the resulting set.

5.  **`yield from ...`**: This is a powerful feature for [generator delegation](https://www.python.org/dev/peps/pep-0380/).
    *   It's used inside a generator function to yield all items from another iterable. In this case, it yields each unique author name, one by one, from the set created by the set comprehension.
    *   It's essentially syntactic sugar for:
        python
        for item in {book.get("author") for book in books if book.get("author")}:
            yield item
        

**In summary:** This code extracts all unique, non-empty author names from a list of book dictionaries and then yields each unique author name one at a time.

---

## Why it does this (Benefits):

1.  **Uniqueness:** The primary reason for using a set comprehension here is to automatically handle duplicate author names. If you have 10 books by "J.K. Rowling", she will only appear once in the output. This is a very common requirement when collecting metadata like author lists.

2.  **Safety & Filtering:**
    *   Using `.get("author")` prevents `KeyError` if some book dictionaries lack an "author" field, making the code more robust.
    *   The `if book.get("author")` clause filters out invalid or missing author entries (e.g., `None` or an empty string), ensuring that only meaningful author names are processed.

3.  **Conciseness & Readability:**
    *   Set comprehensions provide a very Pythonic and compact way to create sets from iterables with optional filtering.
    *   `yield from` offers a clean and efficient way to "flatten" the iteration of an inner iterable (the set) directly into the generator's output, avoiding a manual `for` loop.

4.  **Memory Efficiency (Generator context):**
    *   While the set comprehension *itself* creates a complete set in memory, `yield from` means that the *consumption* of these unique authors happens one at a time. If this generator were part of a larger pipeline, it could contribute to overall memory efficiency by only processing what's needed at each step. However, it's important to note that the *entire set of unique authors* will be materialized in memory before `yield from` starts delegating. If the number of *unique* authors is enormous, this could still be a memory concern, but generally, for unique authors, it's efficient enough.

---

## Example:

python
def get_unique_authors(books_data):
    """
    A generator function that yields unique, non-empty author names
    from a list of book dictionaries.
    """
    yield from {book.get("author") for book in books_data if book.get("author")}

# --- Example Usage ---
books = [
    {"title": "The Hitchhiker's Guide", "author": "Douglas Adams"},
    {"title": "The Restaurant at the End of the Universe", "author": "Douglas Adams"},
    {"title": "Pride and Prejudice", "author": "Jane Austen"},
    {"title": "1984", "author": "George Orwell"},
    {"title": "Animal Farm", "author": "George Orwell"},
    {"title": "Missing Author Book"}, # No 'author' key
    {"title": "Empty Author Book", "author": ""}, # Empty author string
    {"title": "Another by Jane Austen", "author": "Jane Austen"},
    {"title": "A Book by Nobody", "editor": "Someone Else"} # No author key at all
]

# Get the authors
unique_authors = get_unique_authors(books)

print("Unique Authors:")
for author in unique_authors:
    print(f"- {author}")

print("\n--- As a list (to see all at once) ---")
print(list(get_unique_authors(books)))


**Output:**


Unique Authors:
- Jane Austen
- George Orwell
- Douglas Adams

--- As a list (to see all at once) ---
['Jane Austen', 'George Orwell', 'Douglas Adams']


As you can see, "Douglas Adams," "George Orwell," and "Jane Austen" only appear once, and the books with missing or empty author fields were correctly ignored. The order of output from a set is not guaranteed, which is why the order might differ in your run compared to the example (e.g., Douglas Adams might appear first).

None


In [10]:
# Get Llama 3.2 to answer

ollama_via_openai =  OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

def answer_with_llama():
    stream = ollama_via_openai.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

print(answer_with_llama())

**Code Explanation**
===============

The given code is a Python expression that uses generators to extract authors from a list of books.

python
yield from {book.get("author") for book in books if book.get("author")}


Here's a breakdown of how it works:

*   `for ... in` loop: The code iterates over each book in the `books` collection.
*   `if book.get("author")`: For each book, it checks if the "author" key exists and has a non-empty value. If not, the book is skipped.
*   Dictionary comprehension: The expression `{book.get("author") for ...}` creates a dictionary where each key corresponds to an author's name. In Python 3.7+, dictionaries preserve the order of their key-value pairs, so this results in a list-like object (a dictionary, but with O(1) lookups and being iterable).

**Using `yield from`**
------------------------

The expression uses `yield from` to delegate iteration over the resulting dictionary-like object. This means that instead of generating all values at once and storing them in memory, it yields one author's name at a time.

*   Without `yield from`, we would need to collect all authors' names explicitly using e.g., `list()`, which could be inefficient if dealing with large datasets.
*   With `yield from`, the code can generate just the needed values on-the-fly without consuming excessive memory, making it more efficient for handling large amounts of data.

**Return Values**
----------------

The expression does not itself return anything. If you want to use the results in your code, you would typically assign them to a variable:

python
authors = yield from {book.get("author") for book in books if book.get("author")}


In such cases, `yield from` is just helping with iteration over the values in an elegant way.

**Note**: Make sure that all necessary dependencies are imported when you use this type of code. The exact syntax and its effects could vary depending on where it's used.

None
